# VBC: ACO Assigned Beneficiaries by County — Delaware peek

Scope: MSSP ACO penetration in DE counties (county overlay only; HISTORY 002).
Source: `data/sources/vbc/Number_of_ACO_Assigned_Beneficiaries_by_County_2024.zip`.
Reading: small file, read whole, filter `State_Name` = Delaware.

In [3]:
import zipfile
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ZIP = ROOT / "data/sources/vbc/Number_of_ACO_Assigned_Beneficiaries_by_County_2024.zip"
MEMBER = "Number_of_ACO_Assigned_Beneficiaries_by_County_2024.csv"
STATE_COL, STATE = "State_Name", "DELAWARE"  # note: uppercase in VBC county files
with zipfile.ZipFile(ZIP) as z, z.open(MEMBER) as f:
    df = pd.read_csv(f, dtype=str, encoding="latin-1")
de = df[df[STATE_COL] == STATE].copy()
print("all rows:", df.shape, "| DE rows:", de.shape)
de.head()

all rows: (135203, 12) | DE rows: (487, 12)


,Year,ACO_ID,State_Name,County_Name,State_ID,County_ID,AB_Psn_Yrs_ESRD,AB_Psn_Yrs_DIS,AB_Psn_Yrs_AGDU,AB_Psn_Yrs_AGND,Tot_AB_Psn_Yrs,Tot_AB
107,2024,A1001,DELAWARE,KENT,8,0,*,*,.,*,*,*
108,2024,A1001,DELAWARE,NEW CASTLE,8,10,.,.,.,*,*,*
109,2024,A1001,DELAWARE,SUSSEX,8,20,.,.,.,*,*,*
1099,2024,A1026,DELAWARE,KENT,8,0,.,.,.,*,*,*
1100,2024,A1026,DELAWARE,NEW CASTLE,8,10,.,.,.,*,*,*


In [4]:
# ACO penetration per DE county (small-cell masks: '*' non-numeric, '.' zero).
de["Tot_AB_num"] = pd.to_numeric(de["Tot_AB"], errors="coerce")
print("distinct ACOs touching DE:", de["ACO_ID"].nunique())
de.groupby("County_Name").agg(aco_count=("ACO_ID", "nunique"), tot_ab=("Tot_AB_num", "sum"))

distinct ACOs touching DE: 260


,aco_count,tot_ab
County_Name,,
KENT,120,12583.0
NEW CASTLE,183,37569.0
SUSSEX,184,21978.0


## Notes
- Grain: ACO x DE county (Kent, New Castle, Sussex). `Tot_AB` = ACO-assigned FFS beneficiaries residing in that county.
- This is beneficiary *residence*, not DE-based ACOs; label precisely (HISTORY 002).
- Join `ACO_ID` to PY Financial results for ACO name + performance.